# Kinase inhibitors over a dose range

{func}`~mantispy.ds.pki` serves `cpg0008-pki`, a dose-response set from the JUMP pilot {cite:p}`Chandrasekaran_2023`. Fifteen kinase inhibitors were run in U2OS cells over a seven-point dose range, eleven of them at three doses and four at a single dose, across eight plates with 32 to 64 replicate wells per treatment. The object is well-level.

The design spends wells on replicates rather than on a full concentration ladder, so it is the workhorse compound screen here for hit calling and normalization, not for curve fitting. For a screen laid out the other way, many concentrations per compound for fitting a concentration-response curve, see [OASIS pilot](oasis_pilot.ipynb).

**Use it if** your own data is a compound screen with many replicate wells per treatment and you want to call hits, normalize against controls or check for plate artifacts. [Hits, effects and cell loss](../tutorials/compounds/hits.ipynb), [Normalize, select, aggregate](../tutorials/profiles/normalize_and_select.ipynb) and [Plate artifacts and corrections](../tutorials/profiles/artifacts.ipynb) all run on it.

In [1]:
import pandas as pd

import mantispy as mt

In [2]:
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [3]:
adata = mt.ds.pki()
adata

AnnData object with n_obs × n_vars = 3072 × 5839
    obs: 'Metadata_plate_map_name', 'Metadata_broad_sample', 'Metadata_mg_per_ml', 'Metadata_mmoles_per_liter', 'Metadata_solvent', 'Metadata_Plate', 'Metadata_Well', 'Metadata_Site_Count', 'Metadata_Count_Cells', 'Metadata_Count_CellsIncludingEdges', 'Metadata_Count_Cytoplasm', 'Metadata_Count_Nuclei', 'Metadata_Count_NucleiIncludingEdges', 'Metadata_Object_Count', 'Metadata_Barcode', 'Metadata_Supplier', 'Metadata_Supplier_Catalog', 'Metadata_pert_type', 'Metadata_control_type', 'Metadata_CellCount', 'Metadata_SiteCount', 'Metadata_Control', 'Metadata_Compound', 'Metadata_Concentration', 'Metadata_MOA', 'Metadata_Perturbation', 'Metadata_Perturbation_Type'
    var: 'object', 'feature_group', 'feature', 'channel', 'scale', 'angle', 'gray_levels', 'radial_bin', 'params', 'is_feature'
    uns: 'mantispy'
    layers: None (.X)

## The dose design

`Metadata_Compound` names the compound and `Metadata_Concentration` its dose in millimoles per litre. Counting wells per compound and dose shows the ladder each treatment was run on. `Metadata_Control` marks the DMSO wells; the positive controls are treatments and are left unflagged.

In [4]:
obs = adata.obs
design = pd.crosstab(obs["Metadata_Compound"], obs["Metadata_Concentration"])
design.iloc[:8]

Metadata_Concentration,0.004,0.010,0.040,0.200,0.400,1.000,2.000
Metadata_Compound,,,,,,,
BRD-K15179513-001-03-4,0,0,0,0,0,0,32
BRD-K15819326-001-01-6,0,0,0,0,0,0,32
BRD-K40109029-001-06-9,0,0,0,0,0,0,32
BRD-K95785537-001-26-9,0,0,0,0,0,0,32
BRD-U00086672-001-01-9,0,0,0,64,64,64,0
BRD-U00086673-001-01-9,0,0,0,64,64,64,0
BRD-U00086674-001-01-9,0,0,64,64,64,0,0
BRD-U00086675-001-01-9,0,0,64,64,64,0,0


## Cells thin out as the dose rises

Reading cells per well against concentration already shows the cytotoxic arm of the dose response, before any profile distance is computed: the highest doses leave fewer cells than DMSO.

In [5]:
_ = mt.pl.cell_counts(adata, groupby="Metadata_Concentration")